In [1]:
# Mount Google Drive to access the uploaded dataset and save weights
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Copy the zip file from Google Drive to Colab local storage for faster I/O
!cp "/content/drive/MyDrive/roadeye_v4_strict_stratified.zip" "/content/"

# Extract the dataset into a dedicated folder
!unzip -q /content/roadeye_v4_strict_stratified.zip -d /content/dataset_v4

print("✅ Data Unzipped Successfully into /content/dataset_v4!")

✅ Data Unzipped Successfully into /content/dataset_v4!


In [3]:
# Install the ultralytics library for YOLOv8
!pip install ultralytics -q

# Verify installation and system environment
import ultralytics
ultralytics.checks()

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 45.0/112.6 GB disk)


In [4]:
import yaml

# Define the dataset configuration paths
data = {
    'path': '/content/dataset_v4', # Root dataset path
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['Crack', 'Pothole']
}

# Generate the data.yaml file directly inside the dataset folder
with open('/content/dataset_v4/data.yaml', 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("✅ data.yaml generated successfully with isolated paths!")

✅ data.yaml generated successfully with isolated paths!


In [8]:
import yaml

# تم تعديل الـ path ليشمل الفولدر الداخلي الناتج عن فك الضغط
data_config = {
    'path': '/content/dataset_v4/roadeye_v4_strict_stratified',
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': 2,
    'names': ['Crack', 'Pothole']
}

# حفظ الملف في نفس المكان الخارجي عشان خلية التدريب تقرأه
with open('/content/dataset_v4/data.yaml', 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ data.yaml re-generated with the correct nested paths!")

✅ data.yaml re-generated with the correct nested paths!


In [9]:
from ultralytics import YOLO
import os

# =========================================================
# RoadEye - V4 Master Training Cell (Production Ready)
# Goal: Train securely on strictly stratified dataset
# =========================================================

# Initialize from scratch using pre-trained ImageNet weights
model = YOLO('yolov8m.pt')

# Define Google Drive project path for safe storage of weights
project_path = '/content/drive/MyDrive/JapanV4_Clean'
os.makedirs(project_path, exist_ok=True)

print("🚀 Starting RoadEye V4 Training (100 Epochs - Tuned & Secured)...")
print("📦 Saving all weights and logs directly to Google Drive...")

results = model.train(
    # Dataset Configuration
    data='/content/dataset_v4/data.yaml',

    # Core Training Parameters
    epochs=100,
    imgsz=1024,
    batch=6,
    workers=2,

    # Output Configuration (Strict Versioning)
    project=project_path,
    name='v4_clean_run_01', # Explicit run name to prevent log mixing
    exist_ok=False,         # Will throw an error if 'v4_clean_run_01' already exists (Prevents silent overwrites)

    # Reproducibility (Crucial for academic defense)
    seed=42,
    deterministic=True,

    # Optimizer Settings (Tuned for safe convergence)
    optimizer='SGD',
    lr0=0.008,
    lrf=0.1,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    cos_lr=True,

    # Data Augmentation (Refined for road damage physics)
    mosaic=0.85,
    close_mosaic=35,
    mixup=0.0,            # Disabled: MixUp destroys asphalt texture and pothole boundaries
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.15,
    scale=0.6,
    shear=4.0,
    perspective=0.0003,
    flipud=0.0,
    fliplr=0.5,

    # Loss Tuning
    box=7.5,
    cls=0.7,
    dfl=1.5,
    label_smoothing=0.0,

    # Stability & Callbacks
    patience=30,
    save=True,
    save_period=10,
    val=True,

    # Hardware settings
    amp=True,
    cache=False,
    device=0
)

# =========================================================
# FINAL REPORT
# =========================================================
print("\n" + "=" * 60)
print("🎉 V4 TRAINING COMPLETE")
print("=" * 60)
print(f"mAP50      : {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"mAP50-95   : {results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"\n📁 Best weights saved in: {project_path}/v4_clean_run_01/weights/best.pt")

🚀 Starting RoadEye V4 Training (100 Epochs - Tuned & Secured)...
📦 Saving all weights and logs directly to Google Drive...
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=35, cls=0.7, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset_v4/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.008, lrf=0.1, mask_ratio=4, max_det

KeyboardInterrupt: 